# Multi-Layer AI-Generated Image Detection System
### Binary Classification: AI-Generated (1) vs Authentic (0) | Metric: F1 Score

**Architecture:** 7 detection layers → feature concatenation → stacking ensemble (XGBoost + LightGBM + RF → LogReg) → F1-optimised threshold

**Research basis:** Cozzolino et al. CVPR 2024 (CLIP), Synthbuster (Bammey 2024), CNNSpot (Wang et al. 2020), Spectral Learning (CVPR 2025)

**Dataset:** CIFAKE from Kaggle (`birdy654/cifake-real-and-ai-generated-synthetic-images`) — 120K images (60K real + 60K AI)

---

In [3]:
!pip install numpy pandas opencv-python Pillow tqdm scipy PyWavelets matplotlib seaborn scikit-learn xgboost lightgbm joblib torch
from IPython.display import clear_output
clear_output()


In [7]:
# First uninstall the current version
!pip uninstall -y opencv-python

# Install the headless version
!pip install opencv-python-headless

!pip install -U typing_extensions ipywidgets
!pip install kagglehub 

!pip uninstall -y torch torchvision torchaudio
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

from IPython.display import clear_output
clear_output()


In [3]:
# ============================================================
# SECTION 0: SETUP & CONFIGURATION
# ============================================================
import os, sys, warnings, io, hashlib, json
import numpy as np
import pandas as pd
import cv2
from PIL import Image
from PIL.ExifTags import TAGS
from pathlib import Path
from tqdm.auto import tqdm
from scipy.fft import fft2, fftshift
from scipy.signal import find_peaks, convolve2d
from scipy.fft import dct
import pywt
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, classification_report, confusion_matrix,
                             roc_auc_score, precision_score, recall_score)
import xgboost as xgb
import lightgbm as lgb
import joblib

warnings.filterwarnings('ignore')

# ─── Config ───────────────────────────────────────────────
SEED = 42
DATA_DIR = Path("./data")          # ← Change to your dataset path
CACHE_DIR = Path("./feature_cache")
MODEL_DIR = Path("./saved_models")
CACHE_DIR.mkdir(exist_ok=True)
MODEL_DIR.mkdir(exist_ok=True)

IMAGE_SIZE = 512                    # Resize longest edge for feature extraction
VAL_SPLIT = 0.15
N_FOLDS = 5

# Layer toggles — set False to skip (e.g., no GPU → disable CLIP/CNN)
ENABLE_LAYER4_CLIP = True
ENABLE_LAYER5_CNN = True

# ─── Reproducibility ─────────────────────────────────────
import random, torch

def set_seeds(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seeds()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

else:
    print("⚠ No GPU detected — disabling CLIP (Layer 4) and CNN (Layer 5)")
    ENABLE_LAYER4_CLIP = False
    ENABLE_LAYER5_CNN = False

Device: cuda
GPU: NVIDIA RTX A6000
VRAM: 51.0 GB


## Section 1: Dataset Loading & CSV Creation

**Dataset:** CIFAKE from Kaggle  
**Slug:** `birdy654/cifake-real-and-ai-generated-synthetic-images`  
**Structure:** `train/REAL/`, `train/FAKE/`, `test/REAL/`, `test/FAKE/`  

Download and extract to `./data/` before running. Or use kaggle API:
```bash
kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images
unzip cifake-real-and-ai-generated-synthetic-images.zip -d ./data/
```

**Adapt for your NDA dataset:** Just change `DATA_DIR` and ensure your images are in `real/` and `ai/` (or `REAL/` and `FAKE/`) subfolders, OR provide a CSV with `filepath,label` columns.

In [4]:
# ============================================================
# SECTION 1: DATASET LOADING & CSV CREATION
# ============================================================

# ─── Auto-download dataset via KaggleHub ──────────────────
import kagglehub

KAGGLE_DATASET = "birdy654/cifake-real-and-ai-generated-synthetic-images"

try:
    dataset_path = kagglehub.dataset_download(KAGGLE_DATASET)
    DATA_DIR = Path(dataset_path)
    print(f"Dataset downloaded to: {DATA_DIR}")
    
    # Show folder structure
    for item in sorted(DATA_DIR.rglob("*"))[:20]:
        depth = len(item.relative_to(DATA_DIR).parts)
        print(f"  {'  ' * depth}├── {item.name}{'/' if item.is_dir() else ''}")
except Exception as e:
    print(f"KaggleHub download failed: {e}")
    print("Falling back to local DATA_DIR. Make sure dataset is already extracted there.")
    print("Manual download: kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images")

# ─── Helpers ──────────────────────────────────────────────

def collect_images(directory, extensions={".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}):
    """Recursively collect all image file paths from a directory."""
    directory = Path(directory)
    paths = []
    for ext in extensions:
        paths.extend(directory.rglob(f"*{ext}"))
        paths.extend(directory.rglob(f"*{ext.upper()}"))
    return sorted(set(paths))

def build_dataset_csv(data_dir, split="train"):
    """Build CSV with filepath and label columns.
    
    Supports multiple folder naming conventions:
    - REAL/FAKE (CIFAKE style)
    - real/ai (GenImage style)  
    - 0/1 (numeric folders)
    """
    data_dir = Path(data_dir)
    split_dir = data_dir / split
    
    if not split_dir.exists():
        # Try without split subfolder
        split_dir = data_dir
    
    records = []
    
    # Try different naming conventions
    real_names = ["REAL", "real", "Real", "authentic", "0", "nature"]
    fake_names = ["FAKE", "fake", "Fake", "ai", "AI", "1", "ai_generated"]
    
    real_dir = None
    fake_dir = None
    
    for name in real_names:
        candidate = split_dir / name
        if candidate.exists():
            real_dir = candidate
            break
    
    for name in fake_names:
        candidate = split_dir / name
        if candidate.exists():
            fake_dir = candidate
            break
    
    if real_dir is None or fake_dir is None:
        raise FileNotFoundError(
            f"Could not find real/fake subfolders in {split_dir}. "
            f"Expected one of {real_names} and {fake_names}."
        )
    
    real_paths = collect_images(real_dir)
    fake_paths = collect_images(fake_dir)
    
    for p in real_paths:
        records.append({"filepath": str(p), "label": 0})
    for p in fake_paths:
        records.append({"filepath": str(p), "label": 1})
    
    df = pd.DataFrame(records)
    print(f"[{split}] Loaded {len(real_paths)} real + {len(fake_paths)} AI = {len(df)} total")
    print(f"[{split}] Class balance: {df['label'].mean():.1%} AI-generated")
    return df

# ─── Load dataset ─────────────────────────────────────────
try:
    df_train = build_dataset_csv(DATA_DIR, split="train")
    df_test = build_dataset_csv(DATA_DIR, split="test")
    print(f"\nTrain shape: {df_train.shape}")
    print(f"Test shape:  {df_test.shape}")
except FileNotFoundError as e:
    print(f"ERROR: {e}")
    print("\nPlease download the dataset first:")
    print("  pip install kagglehub")
    print("  # Or manually:")
    print("  kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images")
    print("  unzip cifake-real-and-ai-generated-synthetic-images.zip -d ./data/")

100%|██████████| 105M/105M [00:02<00:00, 36.5MB/s] 

Extracting files...


Dataset downloaded to: /home/jovyan/.cache/kagglehub/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images/versions/3
    ├── test/
      ├── FAKE/
        ├── 0 (10).jpg
        ├── 0 (2).jpg
        ├── 0 (3).jpg
        ├── 0 (4).jpg
        ├── 0 (5).jpg
        ├── 0 (6).jpg
        ├── 0 (7).jpg
        ├── 0 (8).jpg
        ├── 0 (9).jpg
        ├── 0.jpg
        ├── 1 (10).jpg
        ├── 1 (2).jpg
        ├── 1 (3).jpg
        ├── 1 (4).jpg
        ├── 1 (5).jpg
        ├── 1 (6).jpg
        ├── 1 (7).jpg
        ├── 1 (8).jpg
[train] Loaded 50000 real + 50000 AI = 100000 total
[train] Class balance: 50.0% AI-generated
[test] Loaded 10000 real + 10000 AI = 20000 total
[test] Class balance: 50.0% AI-generated

Train shape: (100000, 2)
Test shape:  (20000, 2)


## Section 2: Image Loading Utilities

In [5]:
# ============================================================
# SECTION 2: SAFE IMAGE LOADING
# ============================================================

def load_image_rgb(path, target_size=IMAGE_SIZE):
    """Load image as RGB numpy array. Returns None on failure."""
    try:
        img = cv2.imread(str(path), cv2.IMREAD_COLOR)
        if img is None:
            pil_img = Image.open(str(path)).convert("RGB")
            img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
        if img.size == 0:
            return None
        h, w = img.shape[:2]
        if max(h, w) > target_size:
            scale = target_size / max(h, w)
            img = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_AREA)
        return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    except Exception:
        return None

def load_image_gray(path, target_size=IMAGE_SIZE):
    """Load image as grayscale numpy array. Returns None on failure."""
    try:
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            pil_img = Image.open(str(path)).convert("L")
            img = np.array(pil_img)
        if img.size == 0:
            return None
        h, w = img.shape[:2]
        if max(h, w) > target_size:
            scale = target_size / max(h, w)
            img = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_AREA)
        return img
    except Exception:
        return None

def safe_features(func, *args, dim=10):
    """Wrapper: returns np.zeros(dim) if func raises any exception."""
    try:
        result = func(*args)
        return np.nan_to_num(result, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    except Exception:
        return np.zeros(dim, dtype=np.float32)

# Quick test
print("Image loading test:")
test_path = df_train.iloc[0]["filepath"]
rgb = load_image_rgb(test_path)
gray = load_image_gray(test_path)
print(f"  RGB shape: {rgb.shape if rgb is not None else 'FAILED'}")
print(f"  Gray shape: {gray.shape if gray is not None else 'FAILED'}")

Image loading test:
  RGB shape: (32, 32, 3)
  Gray shape: (32, 32)


## Section 3: Layer 1 — Frequency-Domain Spectral Analysis

**Features extracted:** FFT magnitude spectrum, azimuthal radial power spectrum, spectral decay slope, DCT coefficient statistics, Benford's Law deviation, DWT sub-band energy ratios, Synthbuster cross-difference residual filter.

**Research:** Durall et al. 2020, Bammey 2024 (Synthbuster), Karageorgiou CVPR 2025, Frank et al. ICML 2020

In [6]:
# ============================================================
# LAYER 1: FREQUENCY DOMAIN (~186 features)
# ============================================================
L1_DIM = 186

def _azimuthal_average(mag):
    center = np.array(mag.shape) // 2
    Y, X = np.ogrid[:mag.shape[0], :mag.shape[1]]
    R = np.sqrt((X - center[1])**2 + (Y - center[0])**2).astype(int)
    max_r = min(center)
    radial = np.zeros(max_r); counts = np.zeros(max_r)
    mask = R < max_r
    np.add.at(radial, R[mask], mag[mask])
    np.add.at(counts, R[mask], 1)
    counts[counts == 0] = 1
    return radial / counts

def _fft_features_single(channel):
    """FFT features for one channel. Returns 28 features."""
    f = fftshift(fft2(channel.astype(np.float32)))
    mag = np.log1p(np.abs(f))
    stats = [mag.mean(), mag.std(), mag.min(), mag.max(), np.median(mag)]
    
    radial = _azimuthal_average(mag)
    if len(radial) > 10:
        stats.extend(np.percentile(radial, [10, 25, 50, 75, 90, 95, 99]).tolist())
        valid = radial > 0
        if valid.sum() > 5:
            x = np.log(np.arange(1, valid.sum()+1))
            y = np.log(radial[valid][:len(x)] + 1e-10)
            slope, intercept = np.polyfit(x, y, 1)
            stats.extend([slope, intercept])
        else:
            stats.extend([0.0, 0.0])
        mid = len(radial) // 2
        stats.append(radial[mid:].sum() / (radial.sum() + 1e-10))
        q = len(radial) // 4
        for i in range(4):
            stats.append(radial[i*q:(i+1)*q].sum() / (radial.sum() + 1e-10))
        peaks, _ = find_peaks(radial, height=radial.mean())
        stats.append(float(len(peaks)))
        stats.append(float(peaks.std()) if len(peaks) > 1 else 0.0)
    else:
        stats.extend([0.0]*16)
    
    result = np.array(stats[:28], dtype=np.float32)
    if len(result) < 28:
        result = np.pad(result, (0, 28-len(result)))
    return result

def _dct_features(gray):
    """DCT coefficient stats + Benford's Law. Returns 20 features."""
    h, w = gray.shape
    h8, w8 = (h//8)*8, (w//8)*8
    if h8 == 0 or w8 == 0:
        return np.zeros(20, dtype=np.float32)
    gray_f = gray[:h8, :w8].astype(np.float32)
    
    all_coeffs = []
    for i in range(0, h8, 8):
        for j in range(0, w8, 8):
            block = gray_f[i:i+8, j:j+8]
            d = dct(dct(block.T, norm='ortho').T, norm='ortho')
            all_coeffs.append(d.flatten())
    coeffs = np.concatenate(all_coeffs)
    
    stats = [coeffs.mean(), coeffs.std(), np.median(coeffs)]
    stats.extend(np.percentile(coeffs, [5, 25, 75, 95, 99]).tolist())
    cs = coeffs.std()
    stats.append(float(((coeffs-coeffs.mean())**4).mean()/(cs**4+1e-10)) if cs > 0 else 0.0)
    stats.append(float(((coeffs-coeffs.mean())**3).mean()/(cs**3+1e-10)) if cs > 0 else 0.0)
    
    # Benford's Law deviation
    nonzero = np.abs(coeffs[coeffs != 0])
    if len(nonzero) > 100:
        first_digits = (nonzero / (10**np.floor(np.log10(nonzero+1e-10)))).astype(int)
        first_digits = first_digits[(first_digits >= 1) & (first_digits <= 9)]
        if len(first_digits) > 0:
            digit_freq = np.bincount(first_digits, minlength=10)[1:10].astype(float)
            digit_freq /= (digit_freq.sum() + 1e-10)
            benford = np.log10(1 + 1/np.arange(1, 10))
            stats.append(float(np.sum(np.abs(digit_freq - benford))))
        else:
            stats.append(0.0)
    else:
        stats.append(0.0)
    
    result = np.array(stats[:20], dtype=np.float32)
    if len(result) < 20:
        result = np.pad(result, (0, 20-len(result)))
    return result

def _dwt_features(gray):
    """Wavelet sub-band energy ratios. Returns 24 features."""
    features = []
    for wavelet in ['db1', 'db4']:
        try:
            cA, (cH, cV, cD) = pywt.dwt2(gray.astype(np.float32), wavelet)
            energies = [np.sum(x**2) for x in [cA, cH, cV, cD]]
            total = sum(energies) + 1e-10
            features.extend([e/total for e in energies])
            features.extend([np.std(cH), np.std(cV), np.std(cD)])
            features.append(np.mean(np.abs(cD)) / (np.mean(np.abs(cA))+1e-10))
            cA2, (cH2, cV2, cD2) = pywt.dwt2(cA, wavelet)
            features.extend([np.std(cH2), np.std(cV2), np.std(cD2)])
            features.append(np.sum(cD2**2) / (total+1e-10))
        except:
            features.extend([0.0]*12)
    return np.array(features[:24], dtype=np.float32)

def _synthbuster_features(rgb):
    """Cross-difference filter + FFT (Bammey 2024). Returns 30 features."""
    features = []
    for c in range(3):
        ch = rgb[:,:,c].astype(np.float32)
        h, w = ch.shape
        if h < 4 or w < 4:
            features.extend([0.0]*10); continue
        residual = np.zeros_like(ch)
        residual[1:-1,1:-1] = ch[1:-1,1:-1]*4 - ch[:-2,1:-1] - ch[2:,1:-1] - ch[1:-1,:-2] - ch[1:-1,2:]
        f = fftshift(fft2(residual))
        mag = np.log1p(np.abs(f))
        radial = _azimuthal_average(mag)
        peaks, _ = find_peaks(radial, height=radial.mean()+radial.std())
        features.extend([mag.mean(), mag.std(), mag.max(), float(len(peaks)),
                         radial.mean(), radial.std(),
                         np.sum(radial[len(radial)//2:])/(np.sum(radial)+1e-10),
                         residual.mean(), residual.std(), float(np.percentile(np.abs(residual), 95))])
    return np.array(features[:30], dtype=np.float32)

def extract_layer1(image_path):
    """Layer 1: Frequency domain features. Returns array of L1_DIM."""
    gray = load_image_gray(image_path)
    rgb = load_image_rgb(image_path)
    if gray is None or rgb is None:
        return np.zeros(L1_DIM, dtype=np.float32)
    
    parts = [
        safe_features(_fft_features_single, gray, dim=28),                    # 28
        np.concatenate([safe_features(_fft_features_single, rgb[:,:,c], dim=28) for c in range(3)]),  # 84
        safe_features(_dct_features, gray, dim=20),                           # 20
        safe_features(_dwt_features, gray, dim=24),                           # 24
        safe_features(_synthbuster_features, rgb, dim=30),                    # 30
    ]
    result = np.concatenate(parts)
    if len(result) < L1_DIM: result = np.pad(result, (0, L1_DIM-len(result)))
    return np.nan_to_num(result[:L1_DIM]).astype(np.float32)

# ─── Test ──────────────────────────────────────────────────
test_feats = extract_layer1(df_train.iloc[0]["filepath"])
print(f"Layer 1 output: {test_feats.shape}, non-zero: {np.count_nonzero(test_feats)}")

Layer 1 output: (186,), non-zero: 145


## Section 4: Layer 2 — Pixel-Level Forensics

**Features:** ELA (Error Level Analysis), PRNU noise residuals, local noise variance uniformity, saturation/clipping analysis, demosaicking traces.

**Research:** Ferdiansyah et al. 2025, Lukas et al. 2006 (PRNU)

In [7]:
# ============================================================
# LAYER 2: PIXEL FORENSICS (~80 features)
# ============================================================
L2_DIM = 80

def _ela_features(image_path):
    """Error Level Analysis at 3 quality levels. Returns 25 features."""
    original = Image.open(str(image_path)).convert("RGB")
    orig_arr = np.array(original, dtype=np.float32)
    features = []
    for quality in [95, 85, 75]:
        buf = io.BytesIO()
        original.save(buf, "JPEG", quality=quality)
        buf.seek(0)
        resaved = np.array(Image.open(buf), dtype=np.float32)
        min_h = min(orig_arr.shape[0], resaved.shape[0])
        min_w = min(orig_arr.shape[1], resaved.shape[1])
        ela = np.abs(orig_arr[:min_h,:min_w] - resaved[:min_h,:min_w])
        features.extend([ela.mean(), ela.std(), ela.max()])
        features.extend(np.percentile(ela, [25,50,75,95]).tolist())
        # Spatial uniformity: 4×4 grid variance
        h, w = ela.shape[:2]
        grid_means = [ela[i*h//4:(i+1)*h//4, j*w//4:(j+1)*w//4].mean()
                      for i in range(4) for j in range(4)]
        features.append(np.std(grid_means))
    result = np.array(features[:25], dtype=np.float32)
    if len(result) < 25: result = np.pad(result, (0, 25-len(result)))
    return result

def _prnu_features(rgb):
    """PRNU noise residual statistics. Returns 20 features."""
    features = []
    noise_channels = []
    for c in range(3):
        ch = rgb[:,:,c].astype(np.float32)
        denoised = cv2.GaussianBlur(ch, (5,5), 1.5)
        residual = ch - denoised
        noise_channels.append(residual.flatten())
        features.extend([residual.mean(), residual.std(),
                         *np.percentile(residual, [5,25,75,95]).tolist(),
                         np.exp(np.mean(np.log(np.abs(residual)+1e-10)))/(np.mean(np.abs(residual))+1e-10)])
    # Cross-channel correlation
    for i in range(3):
        for j in range(i+1, 3):
            corr = np.corrcoef(noise_channels[i], noise_channels[j])[0,1]
            features.append(corr if not np.isnan(corr) else 0.0)
    result = np.array(features[:20], dtype=np.float32)
    if len(result) < 20: result = np.pad(result, (0, 20-len(result)))
    return result

def _noise_uniformity(gray):
    """Local noise variance uniformity. Returns 15 features."""
    h, w = gray.shape
    bs = max(16, min(h, w) // 8)
    local_vars = []
    for i in range(0, h-bs+1, bs):
        for j in range(0, w-bs+1, bs):
            patch = gray[i:i+bs, j:j+bs].astype(np.float32)
            noise = patch - cv2.GaussianBlur(patch, (3,3), 1.0)
            local_vars.append(noise.var())
    lv = np.array(local_vars) if local_vars else np.array([0.0])
    features = [lv.mean(), lv.std(), lv.min(), lv.max(), np.median(lv),
                lv.std()/(lv.mean()+1e-10)]
    features.extend(np.percentile(lv, [10,25,75,90]).tolist())
    cs = lv.std()
    features.append(float(((lv-lv.mean())**4).mean()/(cs**4+1e-10)) if cs > 0 else 0.0)
    result = np.array(features[:15], dtype=np.float32)
    if len(result) < 15: result = np.pad(result, (0, 15-len(result)))
    return result

def _saturation_features(rgb):
    """Saturation & clipping. Returns 12 features."""
    features = []
    for c in range(3):
        hist = np.histogram(rgb[:,:,c], bins=256, range=(0,256))[0].astype(np.float32)
        hist_norm = hist / (hist.sum()+1e-10)
        features.extend([hist_norm[0], hist_norm[-1],
                         np.abs(np.diff(hist_norm)).mean(), np.std(hist_norm)])
    return np.array(features[:12], dtype=np.float32)

def _demosaicking_features(rgb):
    """Bayer pattern traces. Returns 8 features."""
    features = []
    for c in range(3):
        ch = rgb[:,:,c].astype(np.float32).flatten()
        if len(ch) > 1:
            features.append(float(np.corrcoef(ch[:-1], ch[1:])[0,1]))
        else:
            features.append(0.0)
    for c1 in range(3):
        for c2 in range(c1+1, 3):
            gx1 = cv2.Sobel(rgb[:,:,c1].astype(np.float32), cv2.CV_32F, 1, 0).flatten()
            gx2 = cv2.Sobel(rgb[:,:,c2].astype(np.float32), cv2.CV_32F, 1, 0).flatten()
            corr = np.corrcoef(gx1, gx2)[0,1]
            features.append(corr if not np.isnan(corr) else 0.0)
    green = rgb[:,:,1].astype(np.float32)
    checker = np.zeros_like(green)
    checker[::2,::2] = 1; checker[1::2,1::2] = 1
    features.append(np.sum(green*checker)/(np.sum(green)+1e-10))
    result = np.array(features[:8], dtype=np.float32)
    if len(result) < 8: result = np.pad(result, (0, 8-len(result)))
    return np.nan_to_num(result)

def extract_layer2(image_path):
    """Layer 2: Pixel forensics. Returns array of L2_DIM."""
    rgb = load_image_rgb(image_path)
    gray = load_image_gray(image_path)
    if rgb is None or gray is None:
        return np.zeros(L2_DIM, dtype=np.float32)
    parts = [
        safe_features(_ela_features, image_path, dim=25),
        safe_features(_prnu_features, rgb, dim=20),
        safe_features(_noise_uniformity, gray, dim=15),
        safe_features(_saturation_features, rgb, dim=12),
        safe_features(_demosaicking_features, rgb, dim=8),
    ]
    result = np.concatenate(parts)
    if len(result) < L2_DIM: result = np.pad(result, (0, L2_DIM-len(result)))
    return np.nan_to_num(result[:L2_DIM]).astype(np.float32)

test_feats = extract_layer2(df_train.iloc[0]["filepath"])
print(f"Layer 2 output: {test_feats.shape}, non-zero: {np.count_nonzero(test_feats)}")

Layer 2 output: (80,), non-zero: 64


## Section 5: Layer 3 — EXIF / Metadata Forensics

**Features:** Camera tag presence flags, GPS completeness, software signatures, JPEG quantisation table fingerprint, colour space, file size.

**Note:** AI-generated images typically lack camera-specific EXIF fields entirely. This layer is fragile (metadata can be stripped) but computationally free.

In [8]:
# ============================================================
# LAYER 3: METADATA / EXIF ANALYSIS (~40 features)
# ============================================================
L3_DIM = 40

CAMERA_TAGS = ["Make","Model","LensMake","LensModel","FocalLength",
               "FocalLengthIn35mmFilm","ExposureTime","FNumber",
               "ISOSpeedRatings","WhiteBalance","MeteringMode","Flash",
               "ExposureProgram","ExposureMode","ShutterSpeedValue",
               "ApertureValue","BrightnessValue","ExposureBiasValue"]

AI_KEYWORDS = ["midjourney","stable diffusion","dall-e","dalle","firefly",
               "comfyui","automatic1111","novelai","flux","generative","ai generated"]

def extract_layer3(image_path):
    """Layer 3: Metadata features. Returns array of L3_DIM."""
    try:
        features = []
        # EXIF extraction
        try:
            img = Image.open(str(image_path))
            exif_data = img._getexif() or {}
            decoded = {TAGS.get(k, str(k)): v for k, v in exif_data.items()}
        except:
            decoded = {}
        
        # Camera tag presence (18 features)
        for tag in CAMERA_TAGS:
            features.append(1.0 if tag in decoded else 0.0)
        
        # GPS completeness (1)
        gps_tags = ["GPSLatitude","GPSLongitude","GPSAltitude","GPSTimeStamp"]
        features.append(float(sum(1 for t in gps_tags if any(t in k for k in decoded.keys()))))
        
        # Total tag count (1)
        features.append(float(min(len(decoded), 100)))
        
        # Software analysis (3)
        software = str(decoded.get("Software", "")).lower()
        features.append(1.0 if software else 0.0)
        features.append(float(len(software)))
        features.append(1.0 if any(kw in software for kw in AI_KEYWORDS) else 0.0)
        
        # Dimension consistency (2)
        try:
            img = Image.open(str(image_path))
            aw, ah = img.size
            ew = decoded.get("ExifImageWidth", decoded.get("ImageWidth", 0))
            eh = decoded.get("ExifImageHeight", decoded.get("ImageLength", 0))
            features.append(1.0 if ew and abs(int(ew)-aw) < 5 else 0.0)
            features.append(1.0 if eh and abs(int(eh)-ah) < 5 else 0.0)
        except:
            features.extend([0.0, 0.0])
        
        # Quantisation tables (8)
        try:
            img = Image.open(str(image_path))
            if hasattr(img, "quantization") and img.quantization:
                qt = list(img.quantization.values())[0]
                qt = np.array(list(qt) if isinstance(qt, (list,tuple,bytes)) else [0]*64, dtype=np.float32)[:64]
                if len(qt) < 64: qt = np.pad(qt, (0, 64-len(qt)))
                features.extend([qt.mean(), qt.std(), qt.min(), qt.max(),
                                qt[0], qt[1:4].mean(), qt[-8:].mean(),
                                float(len(img.quantization))])
            else:
                features.extend([0.0]*8)
        except:
            features.extend([0.0]*8)
        
        # Colour space, bit depth, DPI (3)
        try:
            img = Image.open(str(image_path))
            mode_map = {"RGB":0,"RGBA":1,"L":2,"CMYK":3,"P":4}
            features.append(float(mode_map.get(img.mode, 5)))
            bits_map = {"1":1,"L":8,"P":8,"RGB":24,"RGBA":32}
            features.append(float(bits_map.get(img.mode, 0)))
            features.append(float(img.info.get("dpi",(0,))[0]) if "dpi" in img.info else 0.0)
        except:
            features.extend([0.0]*3)
        
        # Thumbnail, file size, orientation (3)
        features.append(1.0 if 513 in (img._getexif() or {}) else 0.0)
        features.append(float(Path(image_path).stat().st_size))
        features.append(1.0 if "Orientation" in decoded else 0.0)
        
        result = np.array(features[:L3_DIM], dtype=np.float32)
        if len(result) < L3_DIM: result = np.pad(result, (0, L3_DIM-len(result)))
        return np.nan_to_num(result)
    except:
        return np.zeros(L3_DIM, dtype=np.float32)

test_feats = extract_layer3(df_train.iloc[0]["filepath"])
print(f"Layer 3 output: {test_feats.shape}, non-zero: {np.count_nonzero(test_feats)}")

Layer 3 output: (40,), non-zero: 10


## Section 6: Layer 4 — CLIP Semantic Features (GPU Required)

**The single strongest detection signal.** CLIP ViT-L/14 features + linear SVM achieves >90% AUC across diverse generators with only ~20 training images (Cozzolino et al. CVPR 2024). Frozen CLIP features contain a natural real-vs-fake separation boundary.

Falls back to ViT-B/32 on low VRAM, or returns zeros if no GPU.

In [10]:
!pip install git+https://github.com/openai/CLIP.git

  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-qcqa6mtu
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-qcqa6mtu
  Resolved https://github.com/openai/CLIP.git to commit d05afc436d78f1c48dc0dbf8e5980a9d471f35f6
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 791.7/791.7 kB 25.1 MB/s eta 0:00:00
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369515 sha256=370ef13e51a37a38f9e53b4a952ec9ba4fa9036b5d015e011367a748fcd71578
  Stored in directory: /tmp/pip-ephem-wheel-cache-u5p_5fp6/wheels/da/2b/4c/d6691fa9597aac8bb85d2ac13b112deb897d5b50f5ad9a37e4
Successfully built clip


In [11]:
# ============================================================
# LAYER 4: CLIP SEMANTIC FEATURES (~768 features)
# ============================================================
L4_DIM = 768

_clip_model = None
_clip_preprocess = None

def _load_clip():
    global _clip_model, _clip_preprocess
    if _clip_model is not None:
        return
    try:
        import clip
        try:
            _clip_model, _clip_preprocess = clip.load("ViT-L/14", device=DEVICE)
        except (RuntimeError, torch.cuda.OutOfMemoryError):
            print("  ViT-L/14 OOM, falling back to ViT-B/32")
            _clip_model, _clip_preprocess = clip.load("ViT-B/32", device=DEVICE)
        _clip_model.eval()
        print(f"  CLIP loaded on {DEVICE}")
    except ImportError:
        print("  ⚠ CLIP not installed. Run: pip install git+https://github.com/openai/CLIP.git")
        _clip_model = None

def extract_layer4(image_path):
    """Layer 4: CLIP features. Returns array of L4_DIM."""
    if not ENABLE_LAYER4_CLIP:
        return np.zeros(L4_DIM, dtype=np.float32)
    try:
        _load_clip()
        if _clip_model is None:
            return np.zeros(L4_DIM, dtype=np.float32)
        pil_img = Image.open(str(image_path)).convert("RGB")
        with torch.no_grad():
            tensor = _clip_preprocess(pil_img).unsqueeze(0).to(DEVICE)
            feats = _clip_model.encode_image(tensor)
            feats = feats / feats.norm(dim=-1, keepdim=True)
        result = feats.cpu().numpy().flatten().astype(np.float32)
        if len(result) < L4_DIM: result = np.pad(result, (0, L4_DIM-len(result)))
        return np.nan_to_num(result[:L4_DIM])
    except:
        return np.zeros(L4_DIM, dtype=np.float32)

if ENABLE_LAYER4_CLIP:
    test_feats = extract_layer4(df_train.iloc[0]["filepath"])
    print(f"Layer 4 (CLIP) output: {test_feats.shape}, non-zero: {np.count_nonzero(test_feats)}")
else:
    print("Layer 4 (CLIP) DISABLED — no GPU")

100%|███████████████████████████████████████| 890M/890M [00:09<00:00, 96.7MiB/s]


  CLIP loaded on cuda
Layer 4 (CLIP) output: (768,), non-zero: 768


## Section 7: Layer 5 — CNN Spatial Artifact Detection (GPU Recommended)

Pretrained EfficientNet-B0 as feature extractor (not classifier). Modified first conv stride=1 to preserve high-frequency traces (Wang et al. CNNSpot 2020).

In [12]:
# ============================================================
# LAYER 5: CNN SPATIAL FEATURES (~256 features)
# ============================================================
L5_DIM = 256

import torch.nn as nn
from torchvision import transforms, models

class CNNExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        first = backbone.features[0][0]
        backbone.features[0][0] = nn.Conv2d(3, first.out_channels,
            kernel_size=first.kernel_size, stride=1, padding=first.padding, bias=False)
        self.features = backbone.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Linear(1280, L5_DIM)
    def forward(self, x):
        return self.proj(self.pool(self.features(x)).flatten(1))

_cnn_model = None
_cnn_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def extract_layer5(image_path):
    """Layer 5: CNN features. Returns array of L5_DIM."""
    global _cnn_model
    if not ENABLE_LAYER5_CNN:
        return np.zeros(L5_DIM, dtype=np.float32)
    try:
        if _cnn_model is None:
            _cnn_model = CNNExtractor().to(DEVICE).eval()
        pil_img = Image.open(str(image_path)).convert("RGB")
        with torch.no_grad():
            tensor = _cnn_transform(pil_img).unsqueeze(0).to(DEVICE)
            feats = _cnn_model(tensor)
        return np.nan_to_num(feats.cpu().numpy().flatten()[:L5_DIM]).astype(np.float32)
    except:
        return np.zeros(L5_DIM, dtype=np.float32)

if ENABLE_LAYER5_CNN:
    test_feats = extract_layer5(df_train.iloc[0]["filepath"])
    print(f"Layer 5 (CNN) output: {test_feats.shape}, non-zero: {np.count_nonzero(test_feats)}")
else:
    print("Layer 5 (CNN) DISABLED — no GPU")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /home/jovyan/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 87.0MB/s]


Layer 5 (CNN) output: (256,), non-zero: 256


## Section 8: Layer 6 — Depth & Geometric Consistency

**Features:** Gradient-based depth proxy, edge density variance, Laplacian focus consistency, shadow/lighting direction analysis.

In [13]:
# ============================================================
# LAYER 6: DEPTH & GEOMETRIC CONSISTENCY (~30 features)
# ============================================================
L6_DIM = 30

def extract_layer6(image_path):
    """Layer 6: Depth & geometry features. Returns array of L6_DIM."""
    gray = load_image_gray(image_path)
    rgb = load_image_rgb(image_path)
    if gray is None or rgb is None:
        return np.zeros(L6_DIM, dtype=np.float32)
    try:
        features = []
        gf = gray.astype(np.float32)
        
        # Gradient statistics
        gx = cv2.Sobel(gf, cv2.CV_32F, 1, 0, ksize=3)
        gy = cv2.Sobel(gf, cv2.CV_32F, 0, 1, ksize=3)
        mag = np.sqrt(gx**2 + gy**2)
        features.extend([mag.mean(), mag.std(), mag.max(), np.median(mag)])
        features.extend(np.percentile(mag, [10,25,75,90,95]).tolist())
        
        # Gradient direction entropy
        angle = np.arctan2(gy, gx+1e-10)
        ahist = np.histogram(angle, bins=36, range=(-np.pi,np.pi))[0].astype(np.float32)
        ahist /= (ahist.sum()+1e-10)
        features.append(float(-np.sum(ahist * np.log(ahist+1e-10))))
        
        # Edge density per quadrant
        edges = cv2.Canny(gray, 50, 150)
        h, w = gray.shape
        qd = [edges[i*h//2:(i+1)*h//2, j*w//2:(j+1)*w//2].mean() for i in range(2) for j in range(2)]
        features.extend(qd)
        features.append(np.std(qd))
        
        # Laplacian focus consistency
        lap = cv2.Laplacian(gf, cv2.CV_32F)
        features.extend([lap.var(), float(np.percentile(np.abs(lap), 95))])
        
        # Shadow/light analysis
        hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
        v = hsv[:,:,2].astype(np.float32)
        features.extend([v.mean(), v.std(), np.median(v)])
        features.append(float(np.arctan2(cv2.Sobel(v,cv2.CV_32F,0,1).mean(),
                                         cv2.Sobel(v,cv2.CV_32F,1,0).mean()+1e-10)))
        qc = [v[i*h//2:(i+1)*h//2, j*w//2:(j+1)*w//2].std() for i in range(2) for j in range(2)]
        features.extend(qc)
        features.append(np.std(qc))
        features.append(v.max() - v.min())
        
        result = np.array(features[:L6_DIM], dtype=np.float32)
        if len(result) < L6_DIM: result = np.pad(result, (0, L6_DIM-len(result)))
        return np.nan_to_num(result)
    except:
        return np.zeros(L6_DIM, dtype=np.float32)

test_feats = extract_layer6(df_train.iloc[0]["filepath"])
print(f"Layer 6 output: {test_feats.shape}, non-zero: {np.count_nonzero(test_feats)}")

Layer 6 output: (30,), non-zero: 27


## Section 9: Layer 7 — Texture Micro-Patterns

**Features:** LBP histograms, GLCM properties, SRM-like high-pass residual filters, colour co-occurrence statistics.

In [14]:
# ============================================================
# LAYER 7: TEXTURE MICRO-PATTERNS (~80 features)
# ============================================================
L7_DIM = 80

def _lbp_features(gray):
    h, w = gray.shape
    g = gray.astype(np.int32)
    lbp = np.zeros_like(g, dtype=np.uint8)
    for i, (dy,dx) in enumerate([(-1,-1),(-1,0),(-1,1),(0,1),(1,1),(1,0),(1,-1),(0,-1)]):
        lbp |= ((np.roll(np.roll(g,dy,0),dx,1) >= g).astype(np.uint8) << i)
    lbp = lbp[1:-1,1:-1]
    hist = np.histogram(lbp, bins=26, range=(0,256))[0].astype(np.float32)
    hist /= (hist.sum()+1e-10)
    features = list(hist)
    features.extend([float(-np.sum(hist*np.log(hist+1e-10))), float(hist.max()),
                     float(hist.std()), float(np.sum(hist[:3]))])
    return np.array(features[:30], dtype=np.float32)

def _glcm_features(gray):
    q = (gray / 8).astype(np.int32)
    levels = 32
    features = []
    for dx,dy in [(1,0),(0,1),(1,1),(1,-1)]:
        glcm = np.zeros((levels,levels), dtype=np.float32)
        h, w = q.shape
        for i in range(max(0,-dy), min(h, h-dy)):
            for j in range(max(0,-dx), min(w, w-dx)):
                v1, v2 = min(q[i,j],levels-1), min(q[i+dy,j+dx],levels-1)
                glcm[v1,v2] += 1
        glcm /= (glcm.sum()+1e-10)
        idx_i, idx_j = np.mgrid[:levels,:levels]
        features.extend([
            float(np.sum((idx_i-idx_j)**2 * glcm)),
            float(np.sum(glcm**2)),
            float(np.sum(glcm/(1+np.abs(idx_i-idx_j)))),
            float(np.sum(idx_i*idx_j*glcm) - np.sum(idx_i*glcm.sum(1,keepdims=True))*np.sum(idx_j*glcm.sum(0,keepdims=True))),
            float(-np.sum(glcm*np.log(glcm+1e-10))),
        ])
    return np.array(features[:20], dtype=np.float32)

def _srm_features(gray):
    gf = gray.astype(np.float32)
    kernels = [
        np.array([[-1,1,0]], dtype=np.float32),
        np.array([[-1],[1],[0]], dtype=np.float32),
        np.array([[1,-2,1]], dtype=np.float32),
        np.array([[-1,3,-3,1]], dtype=np.float32),
        np.array([[-1,2,-1],[2,-4,2],[-1,2,-1]], dtype=np.float32)/4,
        np.array([[0,0,0],[-1,2,-1],[0,0,0]], dtype=np.float32),
    ]
    features = []
    for k in kernels:
        r = convolve2d(gf, k, mode='valid', boundary='symm')
        features.extend([r.mean(), r.std(), float(np.percentile(np.abs(r), 95))])
    return np.array(features[:18], dtype=np.float32)

def _colour_cooccurrence(rgb):
    features = []
    for c in range(3):
        ch = rgb[:,:,c].astype(np.float32)
        dh = np.abs(ch[:,1:] - ch[:,:-1])
        dv = np.abs(ch[1:,:] - ch[:-1,:])
        features.extend([dh.mean(), dh.std(), dv.mean(), dv.std()])
    return np.array(features[:12], dtype=np.float32)

def extract_layer7(image_path):
    """Layer 7: Texture features. Returns array of L7_DIM."""
    gray = load_image_gray(image_path, target_size=256)
    rgb = load_image_rgb(image_path, target_size=256)
    if gray is None or rgb is None:
        return np.zeros(L7_DIM, dtype=np.float32)
    parts = [
        safe_features(_lbp_features, gray, dim=30),
        safe_features(_glcm_features, gray, dim=20),
        safe_features(_srm_features, gray, dim=18),
        safe_features(_colour_cooccurrence, rgb, dim=12),
    ]
    result = np.concatenate(parts)
    if len(result) < L7_DIM: result = np.pad(result, (0, L7_DIM-len(result)))
    return np.nan_to_num(result[:L7_DIM]).astype(np.float32)

test_feats = extract_layer7(df_train.iloc[0]["filepath"])
print(f"Layer 7 output: {test_feats.shape}, non-zero: {np.count_nonzero(test_feats)}")

Layer 7 output: (80,), non-zero: 79


## Section 10: Full Feature Extraction Pipeline

Runs all 7 layers on the entire dataset, concatenates into a single matrix, and caches to disk.

In [15]:
# ============================================================
# FULL FEATURE EXTRACTION
# ============================================================

LAYER_REGISTRY = [
    ("L1_freq",    extract_layer1,  L1_DIM),
    ("L2_pixel",   extract_layer2,  L2_DIM),
    ("L3_meta",    extract_layer3,  L3_DIM),
    ("L4_clip",    extract_layer4,  L4_DIM),
    ("L5_cnn",     extract_layer5,  L5_DIM),
    ("L6_depth",   extract_layer6,  L6_DIM),
    ("L7_texture", extract_layer7,  L7_DIM),
]

TOTAL_DIM = sum(dim for _, _, dim in LAYER_REGISTRY)
print(f"Total feature dimension: {TOTAL_DIM}")
for name, _, dim in LAYER_REGISTRY:
    print(f"  {name}: {dim}")

def extract_all_features(image_path):
    """Extract features from all layers for a single image."""
    parts = [func(image_path) for _, func, _ in LAYER_REGISTRY]
    return np.concatenate(parts).astype(np.float32)

def extract_batch(df, cache_name=None):
    """Extract features for entire dataframe. Caches to disk."""
    if cache_name:
        cache_path = CACHE_DIR / f"{cache_name}.npy"
        if cache_path.exists():
            cached = np.load(cache_path)
            if cached.shape[0] == len(df):
                print(f"Loaded cached features: {cache_path} ({cached.shape})")
                return cached
    
    features = np.zeros((len(df), TOTAL_DIM), dtype=np.float32)
    for i, row in enumerate(tqdm(df.itertuples(), total=len(df), desc=f"Extracting [{cache_name}]")):
        features[i] = extract_all_features(row.filepath)
    
    if cache_name:
        np.save(CACHE_DIR / f"{cache_name}.npy", features)
        print(f"Cached to {CACHE_DIR / cache_name}.npy")
    
    nz = np.count_nonzero(features, axis=0)
    print(f"Feature matrix: {features.shape}")
    print(f"Columns with >50% non-zero: {(nz > len(df)*0.5).sum()}/{TOTAL_DIM}")
    return features

Total feature dimension: 1440
  L1_freq: 186
  L2_pixel: 80
  L3_meta: 40
  L4_clip: 768
  L5_cnn: 256
  L6_depth: 30
  L7_texture: 80


In [16]:
# ============================================================
# RUN EXTRACTION (this takes a while — results are cached)
# ============================================================

# Train/val split
train_paths, val_paths, train_labels, val_labels = train_test_split(
    df_train["filepath"].values, df_train["label"].values,
    test_size=VAL_SPLIT, stratify=df_train["label"].values, random_state=SEED
)

df_train_split = pd.DataFrame({"filepath": train_paths, "label": train_labels})
df_val_split = pd.DataFrame({"filepath": val_paths, "label": val_labels})

print(f"Train: {len(df_train_split)} | Val: {len(df_val_split)}")
print(f"Train class balance: {train_labels.mean():.1%} AI")
print(f"Val class balance: {val_labels.mean():.1%} AI")

# Extract features
X_train = extract_batch(df_train_split, cache_name="train_features")
X_val = extract_batch(df_val_split, cache_name="val_features")
y_train = train_labels
y_val = val_labels

Train: 85000 | Val: 15000
Train class balance: 50.0% AI
Val class balance: 50.0% AI


Extracting [train_features]:   0%|          | 0/85000 [00:00<?, ?it/s]

Cached to feature_cache/train_features.npy
Feature matrix: (85000, 1440)
Columns with >50% non-zero: 1346/1440


Extracting [val_features]:   0%|          | 0/15000 [00:00<?, ?it/s]

Cached to feature_cache/val_features.npy
Feature matrix: (15000, 1440)
Columns with >50% non-zero: 1346/1440


## Section 11: Model Training — Stacking Ensemble

**Level 1:** XGBoost + LightGBM + Random Forest (trained with out-of-fold predictions to prevent data leakage)  
**Level 2:** Logistic Regression meta-learner (trained on OOF predictions)

In [17]:
# ============================================================
# STACKING ENSEMBLE TRAINING
# ============================================================

# Scale features
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

# Base models
base_models = {
    "xgb": xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        eval_metric="logloss", random_state=SEED, n_jobs=-1, verbosity=0
    ),
    "lgb": lgb.LGBMClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0,
        random_state=SEED, n_jobs=-1, verbose=-1
    ),
    "rf": RandomForestClassifier(
        n_estimators=300, max_depth=15, min_samples_leaf=5,
        random_state=SEED, n_jobs=-1
    ),
}

# Out-of-fold predictions for stacking
print("Training stacking ensemble with 5-fold OOF predictions...\n")
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_preds = np.zeros((len(X_train_scaled), len(base_models)))

for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X_train_scaled, y_train)):
    print(f"Fold {fold_idx+1}/{N_FOLDS}")
    X_tr, X_vl = X_train_scaled[tr_idx], X_train_scaled[val_idx]
    y_tr, y_vl = y_train[tr_idx], y_train[val_idx]
    
    for m_idx, (name, model) in enumerate(base_models.items()):
        import sklearn.base
        fold_model = sklearn.base.clone(model)
        
        if name == "xgb":
            fold_model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)], verbose=False)
        elif name == "lgb":
            fold_model.fit(X_tr, y_tr, eval_set=[(X_vl, y_vl)])
        else:
            fold_model.fit(X_tr, y_tr)
        
        oof_preds[val_idx, m_idx] = fold_model.predict_proba(X_vl)[:, 1]
    
    # Fold metrics
    fold_f1s = {name: f1_score(y_vl, (oof_preds[val_idx, i] >= 0.5).astype(int))
                for i, name in enumerate(base_models.keys())}
    print(f"  F1: {fold_f1s}")

# Train meta-model
meta_model = LogisticRegression(C=1.0, max_iter=1000, random_state=SEED)
meta_model.fit(oof_preds, y_train)

# Retrain base models on full training data
print("\nRetraining base models on full training data...")
for name, model in base_models.items():
    model.fit(X_train_scaled, y_train)
    print(f"  {name} trained")

# OOF meta-performance
meta_oof_probs = meta_model.predict_proba(oof_preds)[:, 1]
oof_f1 = f1_score(y_train, (meta_oof_probs >= 0.5).astype(int))
print(f"\nMeta-model OOF F1 (threshold=0.5): {oof_f1:.4f}")

Training stacking ensemble with 5-fold OOF predictions...

Fold 1/5
  F1: {'xgb': 0.9757013734006339, 'lgb': 0.9742685935847727, 'rf': 0.9453138737471426}
Fold 2/5
  F1: {'xgb': 0.9754348848142924, 'lgb': 0.9742685935847727, 'rf': 0.9455969146263075}
Fold 3/5
  F1: {'xgb': 0.9773140277859195, 'lgb': 0.976075993901724, 'rf': 0.9440187189236618}
Fold 4/5
  F1: {'xgb': 0.9728331177231565, 'lgb': 0.9741658329900548, 'rf': 0.941872036527542}
Fold 5/5
  F1: {'xgb': 0.9743951139299977, 'lgb': 0.9742986531788508, 'rf': 0.9440028057049333}

Retraining base models on full training data...
  xgb trained
  lgb trained
  rf trained

Meta-model OOF F1 (threshold=0.5): 0.9749


## Section 12: F1-Optimal Threshold Tuning

**Critical step.** The default threshold of 0.5 is almost never optimal for F1. We sweep [0.15, 0.85] and select the threshold that maximises F1 on the validation set.

In [18]:
# ============================================================
# THRESHOLD OPTIMISATION
# ============================================================

# Get validation predictions
val_base_preds = np.zeros((len(X_val_scaled), len(base_models)))
for m_idx, (name, model) in enumerate(base_models.items()):
    val_base_preds[:, m_idx] = model.predict_proba(X_val_scaled)[:, 1]
val_probs = meta_model.predict_proba(val_base_preds)[:, 1]

# Sweep thresholds
thresholds = np.arange(0.15, 0.85, 0.005)
sweep_results = []
best_f1, best_thresh = 0, 0.5

for t in thresholds:
    preds = (val_probs >= t).astype(int)
    f1 = f1_score(y_val, preds, zero_division=0)
    prec = precision_score(y_val, preds, zero_division=0)
    rec = recall_score(y_val, preds, zero_division=0)
    sweep_results.append({"threshold": t, "f1": f1, "precision": prec, "recall": rec})
    if f1 > best_f1:
        best_f1, best_thresh = f1, t

print("=" * 50)
print(f"OPTIMAL THRESHOLD: {best_thresh:.3f}")
print(f"BEST F1 SCORE:     {best_f1:.4f}")
print(f"PRECISION:         {precision_score(y_val, (val_probs >= best_thresh).astype(int)):.4f}")
print(f"RECALL:            {recall_score(y_val, (val_probs >= best_thresh).astype(int)):.4f}")
print("=" * 50)

OPTIMAL THRESHOLD: 0.435
BEST F1 SCORE:     0.9774
PRECISION:         0.9724
RECALL:            0.9824


## Section 13: Evaluation — Confusion Matrix, Threshold Sweep, Score Distribution

In [19]:
# ============================================================
# EVALUATION & PLOTS
# ============================================================

y_val_pred = (val_probs >= best_thresh).astype(int)

# Classification report
print(classification_report(y_val, y_val_pred, target_names=["Real (0)", "AI-Generated (1)"]))

# Confusion matrix
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cm = confusion_matrix(y_val, y_val_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Real","AI"], yticklabels=["Real","AI"], ax=axes[0])
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_title(f"Confusion Matrix (thresh={best_thresh:.3f})")

# Threshold sweep
sweep_df = pd.DataFrame(sweep_results)
axes[1].plot(sweep_df["threshold"], sweep_df["f1"], label="F1", linewidth=2)
axes[1].plot(sweep_df["threshold"], sweep_df["precision"], "--", label="Precision")
axes[1].plot(sweep_df["threshold"], sweep_df["recall"], "--", label="Recall")
axes[1].axvline(best_thresh, color="red", linestyle=":", label=f"Optimal ({best_thresh:.3f})")
axes[1].set_xlabel("Threshold"); axes[1].set_ylabel("Score")
axes[1].set_title("Threshold Sweep"); axes[1].legend(); axes[1].grid(alpha=0.3)

# Score distribution
axes[2].hist(val_probs[y_val==0], bins=50, alpha=0.6, label="Real", color="blue")
axes[2].hist(val_probs[y_val==1], bins=50, alpha=0.6, label="AI-Generated", color="red")
axes[2].axvline(best_thresh, color="black", linestyle="--", label=f"Threshold ({best_thresh:.3f})")
axes[2].set_xlabel("Predicted Probability"); axes[2].set_ylabel("Count")
axes[2].set_title("Score Distribution"); axes[2].legend()

plt.tight_layout()
plt.savefig("evaluation_plots.png", dpi=150, bbox_inches="tight")
plt.show()
print("Plots saved to evaluation_plots.png")

                  precision    recall  f1-score   support

        Real (0)       0.98      0.97      0.98      7500
AI-Generated (1)       0.97      0.98      0.98      7500

        accuracy                           0.98     15000
       macro avg       0.98      0.98      0.98     15000
    weighted avg       0.98      0.98      0.98     15000

Plots saved to evaluation_plots.png


## Section 14: Inference on Test Set → submission.csv

In [20]:
# ============================================================
# INFERENCE & SUBMISSION
# ============================================================

# Extract test features
X_test = extract_batch(df_test, cache_name="test_features")
X_test_scaled = scaler.transform(X_test)

# Predict
test_base_preds = np.zeros((len(X_test_scaled), len(base_models)))
for m_idx, (name, model) in enumerate(base_models.items()):
    test_base_preds[:, m_idx] = model.predict_proba(X_test_scaled)[:, 1]
test_probs = meta_model.predict_proba(test_base_preds)[:, 1]
test_preds = (test_probs >= best_thresh).astype(int)

# Create submission
submission = pd.DataFrame({
    "filename": [Path(p).name for p in df_test["filepath"]],
    "prediction": test_preds,
    "probability": test_probs,
})
submission.to_csv("submission.csv", index=False)

print(f"Submission saved: submission.csv")
print(f"Total predictions: {len(submission)}")
print(f"Predicted AI: {test_preds.sum()} ({test_preds.mean():.1%})")
print(f"Predicted Real: {(1-test_preds).sum()} ({(1-test_preds).mean():.1%})")

# If test labels are available, compute F1
if "label" in df_test.columns:
    test_f1 = f1_score(df_test["label"].values, test_preds)
    print(f"\nTest F1 Score: {test_f1:.4f}")
    print(classification_report(df_test["label"].values, test_preds, 
                                target_names=["Real","AI-Generated"]))

Extracting [test_features]:   0%|          | 0/20000 [00:00<?, ?it/s]

Cached to feature_cache/test_features.npy
Feature matrix: (20000, 1440)
Columns with >50% non-zero: 1346/1440
Submission saved: submission.csv
Total predictions: 20000
Predicted AI: 10069 (50.3%)
Predicted Real: 9931 (49.7%)

Test F1 Score: 0.9753
              precision    recall  f1-score   support

        Real       0.98      0.97      0.98     10000
AI-Generated       0.97      0.98      0.98     10000

    accuracy                           0.98     20000
   macro avg       0.98      0.98      0.98     20000
weighted avg       0.98      0.98      0.98     20000



## Section 15: Save All Model Artifacts

In [21]:
# ============================================================
# SAVE MODELS
# ============================================================

MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(scaler, MODEL_DIR / "scaler.pkl")
joblib.dump(meta_model, MODEL_DIR / "meta_model.pkl")
for name, model in base_models.items():
    joblib.dump(model, MODEL_DIR / f"base_{name}.pkl")

threshold_config = {
    "optimal_threshold": float(best_thresh),
    "val_f1": float(best_f1),
}
with open(MODEL_DIR / "threshold.json", "w") as f:
    json.dump(threshold_config, f, indent=2)

print(f"All models saved to {MODEL_DIR}/")
print(f"Files: {list(MODEL_DIR.glob('*'))}")

# ─── Quick-predict function for new images ─────────────────
def predict_image(image_path):
    """Predict a single image. Returns dict with prediction and probability."""
    feats = extract_all_features(image_path).reshape(1, -1)
    feats_scaled = scaler.transform(feats)
    base_preds = np.array([[m.predict_proba(feats_scaled)[0, 1] for m in base_models.values()]])
    prob = meta_model.predict_proba(base_preds)[0, 1]
    pred = 1 if prob >= best_thresh else 0
    return {"prediction": pred, "probability": round(float(prob), 4),
            "label": "AI-Generated" if pred == 1 else "Authentic"}

# Test it
result = predict_image(df_test.iloc[0]["filepath"])
print(f"\nSingle image prediction test: {result}")

All models saved to saved_models/
Files: [PosixPath('saved_models/meta_model.pkl'), PosixPath('saved_models/scaler.pkl'), PosixPath('saved_models/base_rf.pkl'), PosixPath('saved_models/base_xgb.pkl'), PosixPath('saved_models/threshold.json'), PosixPath('saved_models/base_lgb.pkl')]

Single image prediction test: {'prediction': 0, 'probability': 0.0079, 'label': 'Authentic'}
